In [ ]:
import os
import ast
import json
import pickle
import warnings
import numpy as np
import xarray as xr
import proplot as pplt
warnings.filterwarnings('ignore')
pplt.rc.update({
    'savefig.dpi':900,
    'savefig.bbox':'tight',
    'savefig.pad_inches':0.02,
    'tick.minor':False,
    'font.size':9,
    'label.size':9,
    'tick.labelsize':9,
    'title.size':9,
    'abc.size':9,
    'legend.fontsize':9,
    'suptitle.size':9,
    'leftlabelsize':9,
    'toplabelsize':9,
    'leftlabel.weight':'normal',
    'toplabel.weight':'normal',
    'reso':'xx-hi'})
import cartopy.crs as ccrs

In [ ]:
with open('../scripts/configs.json','r',encoding='utf-8') as f:
    CONFIGS = json.load(f)
SPLITSDIR    = CONFIGS['filepaths']['splits']
WEIGHTSDIR   = CONFIGS['filepaths']['weights']
MODELSDIR    = CONFIGS['filepaths']['models']
PREDSDIR     = CONFIGS['filepaths']['predictions']
MODELS       = CONFIGS['experiments']
FIELDVARS    = MODELS['sr']['runs']['sr_gauss']['fieldvars']
NNSEEDS      = MODELS['nn']['seeds']
OPTIMIZEDEQS = MODELS['sr']['optimizedeqs']
PODRUNS      = MODELS['pod']['runs']
NNRUNS       = MODELS['nn']['runs']
ORDER        = ['pod_bl','nn_bl','nn_full','nn_nonparam','nn_gauss','sr_lo','sr_bl','sr_med','sr_hi']
SPLIT        = 'test'
NBINS        = 20
MINSAMPLES   = 50
SRFUNCTIONS  = {
    'cube':lambda x:x**3,
    'square':lambda x:x**2,
    'neg':lambda x:-x,
    'sqrt':np.sqrt,
    'exp':np.exp,
    'log':np.log,
    'abs':np.abs,
    'max':np.maximum,
    'min':np.minimum}
COLORS = {}
LABELS = {}
for name,config in {**PODRUNS,**NNRUNS,**OPTIMIZEDEQS}.items():
    COLORS[name] = config['color']
    LABELS[name] = config['description']
RAWDIR = CONFIGS['filepaths']['raw']

In [ ]:
def kernel_integrate(fields,weights,dsig,mask=None):
    w = fields*weights[None,:,:]*dsig[None,None,:]
    if mask is not None: w = w*mask[:,None,:]
    return w.sum(axis=2)

def physical_prediction(output):
    return np.expm1(tpstd*np.maximum(0.0,np.asarray(output,dtype=float)))

def eval_form(form,variables,constants):
    ns = dict(SRFUNCTIONS); ns.update(variables); ns.update(constants)
    return np.asarray(eval(form,{'__builtins__':{}},ns),dtype=float)

def used_predictors(form,candidates):
    names = {n.id for n in ast.walk(ast.parse(form,mode='eval')) if isinstance(n,ast.Name)}
    return [c for c in candidates if c in names]

def r2(obs,pred):
    mask = np.isfinite(obs)&np.isfinite(pred)
    o,p  = obs[mask],pred[mask]
    return 1-np.sum((o-p)**2)/np.sum((o-o.mean())**2)

def bin_1d(x,z,nbins=NBINS,minsamples=MINSAMPLES,plo=1,phi=99):
    mask = np.isfinite(x)&np.isfinite(z)
    x,z  = x[mask],z[mask]
    lo,hi = np.percentile(x,[plo,phi])
    edges = np.linspace(lo,hi,nbins+1)
    xidxs = np.clip(np.digitize(x,edges)-1,0,nbins-1)
    means = np.full(nbins,np.nan); counts = np.zeros(nbins,dtype=int)
    for i in range(nbins):
        idx = xidxs==i; counts[i] = idx.sum()
        if counts[i]>=minsamples: means[i] = z[idx].mean()
    return 0.5*(edges[:-1]+edges[1:]),means,counts

In [ ]:
with open(os.path.join(SPLITSDIR,'stats.json'),'r',encoding='utf-8') as f:
    STATS = json.load(f)
tpmean = float(STATS['tp_mean'])
tpstd  = float(STATS['tp_std'])

with xr.open_dataset(os.path.join(SPLITSDIR,f'norm_{SPLIT}.h5'),engine='h5netcdf') as ds:
    ntime = ds.sizes['time']; nlat = ds.sizes['lat']; nlon = ds.sizes['lon']
    nsig  = ds.sizes.get('sig',1)
    lat   = ds['lat'].values; lon = ds['lon'].values; dsig = ds['dsig'].values
    farrs      = [ds[v].transpose('time','lat','lon','sig').values.reshape(-1,nsig) for v in FIELDVARS]
    fieldstack = np.stack(farrs,axis=1)
    surfmask   = (ds['surfmask'].transpose('time','lat','lon','sig').values.reshape(-1,nsig)
                  if 'surfmask' in ds else None)
    def getflat(da):
        if 'time' in da.dims: return da.transpose('time','lat','lon').values.ravel()
        return np.tile(da.values,(ntime,1,1)).ravel()
    blnorm  = getflat(ds['bl']); lfnorm  = getflat(ds['lf'])
    shfnorm = getflat(ds['shf']); lhfnorm = getflat(ds['lhf'])

kwlist = []
for seed in NNSEEDS:
    wpath = os.path.join(WEIGHTSDIR,f'nn_gauss_{seed}_weights.nc')
    if os.path.exists(wpath):
        with xr.open_dataset(wpath,engine='h5netcdf') as wds:
            kwlist.append(wds['k'].values)
ki = (np.mean([kernel_integrate(fieldstack,kw,dsig,surfmask) for kw in kwlist],axis=0)
      if kwlist else fieldstack.mean(axis=2))
rhk,thetaek,thetaestark = ki[:,0],ki[:,1],ki[:,2]

with xr.open_dataset(os.path.join(SPLITSDIR,f'{SPLIT}.h5'),engine='h5netcdf') as ds:
    obsflat = ds.tp.transpose('time','lat','lon').values.ravel()
    sdoraw  = getflat(ds['sdo'])

with open(os.path.join(MODELSDIR,'sr','optimized_equations.pkl'),'rb') as f:
    SR_REGISTRY = pickle.load(f)

SEMAP = SEFLAT = None
sepath = os.path.join(RAWDIR,'ERA5_surface_elevation.nc')
if os.path.exists(sepath):
    with xr.open_dataset(sepath,engine='netcdf4') as ds:
        sevar = list(ds.data_vars)[0]; se = ds[sevar]
        if 'time' in se.dims: se = se.isel(time=0)
        se    = se.interp(lat=lat,lon=lon,method='linear')
        SEMAP = se.values.astype(np.float32)
    SEFLAT = np.tile(SEMAP,(ntime,1,1)).ravel()
    print(f'Surface elevation: {SEMAP.min():.0f}\u2013{SEMAP.max():.0f} m')
SDOMAP = sdoraw.reshape(ntime,nlat,nlon)[0]

In [ ]:
VARS = {'bl':blnorm,'rh':rhk,'thetae':thetaek,'thetaestar':thetaestark,
        'lf':lfnorm,'shf':shfnorm,'lhf':lhfnorm}

MODELPRED = {}
for name,cfg in OPTIMIZEDEQS.items():
    entry     = SR_REGISTRY.get(name,{})
    form      = entry.get('form',cfg['form'])
    constants = entry.get('constants',cfg['init'])
    pnames    = used_predictors(form,VARS.keys())
    MODELPRED[name] = physical_prediction(eval_form(form,{p:VARS[p] for p in pnames},constants))

def load_pred(name,split=SPLIT):
    path = os.path.join(PREDSDIR,f'{name}_{split}_predictions.nc')
    if not os.path.exists(path): return None
    with xr.open_dataset(path,engine='h5netcdf') as ds:
        da = ds['tp'].load()
    if 'seed' in da.dims: da = da.mean('seed')
    pred = da.transpose('time','lat','lon').values.ravel()
    return pred if pred.shape[0]==obsflat.shape[0] else None

for name in {**PODRUNS,**NNRUNS}:
    pred = load_pred(name)
    if pred is not None: MODELPRED[name] = pred

valid = np.isfinite(obsflat)
for arr in VARS.values(): valid &= np.isfinite(arr)
print('Loaded:', sorted(MODELPRED.keys()))
for name,pred in MODELPRED.items():
    print(f'  {LABELS.get(name,name):20s}  R\u00b2={r2(obsflat[valid],pred[valid]):.3f}')

In [ ]:
def tomap(flat):
    return np.nanmean(flat.reshape(ntime,nlat,nlon),axis=0)

obsmap    = tomap(obsflat)
residmaps = {name: tomap(MODELPRED[name])-obsmap for name in MODELPRED}

In [ ]:
import cartopy.crs as ccrs
names     = [n for n in ORDER if n in residmaps]
allpanels = ['obs']+names
ncols = min(3,len(allpanels)); nrows = -(-len(allpanels)//ncols)
fig,axs = pplt.subplots(nrows=nrows,ncols=ncols,proj='cyl',refwidth=2.5,share=True)
axsf    = np.atleast_1d(axs).ravel()
mc = mr = None
for ax,panel in zip(axsf,allpanels):
    if panel=='obs':
        mc = ax.pcolormesh(lon,lat,obsmap,cmap='Blues',vmin=0,vmax=2,extend='max',
                           transform=ccrs.PlateCarree())
        ax.format(title='Observed')
    else:
        mr = ax.pcolormesh(lon,lat,residmaps[panel],cmap='DryWet',vmin=-0.5,vmax=0.5,
                           extend='both',transform=ccrs.PlateCarree())
        ax.format(title=LABELS.get(panel,panel))
    ax.format(latlim=(float(lat.min()),float(lat.max())),
              lonlim=(float(lon.min()),float(lon.max())),
              coast=True,borders=False,grid=True,gridminor=False)
for ax in axsf[len(allpanels):]: ax.set_visible(False)
if mc: fig.colorbar(mc,loc='b',label='Observed Precipitation (mm)',width=0.15,length=0.3,col=1)
if mr: fig.colorbar(mr,loc='b',label='Predicted \u2212 Observed (mm)',
                    width=0.15,length=0.5,col=(2,ncols))
pplt.show()
fig.save('../figs/fig_S4.jpg')

In [ ]:
srnames = [n for n in ['sr_bl','sr_lo','sr_med','sr_hi'] if n in residmaps]
fig,axs = pplt.subplots(nrows=1,ncols=len(srnames),proj='cyl',refwidth=2.5,share=True)
axsf    = np.atleast_1d(axs).ravel()
m = None
for ax,name in zip(axsf,srnames):
    m = ax.pcolormesh(lon,lat,residmaps[name],cmap='DryWet',vmin=-0.5,vmax=0.5,
                      extend='both',transform=ccrs.PlateCarree())
    ax.contour(lon,lat,SDOMAP,levels=[100,300,500],colors='k',linewidths=0.5,
               transform=ccrs.PlateCarree())
    ax.format(title=LABELS.get(name,name),
              latlim=(float(lat.min()),float(lat.max())),
              lonlim=(float(lon.min()),float(lon.max())),
              coast=True,borders=False,grid=True,gridminor=False)
fig.colorbar(m,loc='b',label='Predicted \u2212 Observed Precipitation (mm)',width=0.15,length=0.7)
pplt.show()
fig.save('../figs/fig_S5.jpg')

In [ ]:
panels = [('Std dev of orography (m)',SDOMAP)]
if SEMAP is not None: panels.append(('Surface elevation (m)',SEMAP))
fig,axs = pplt.subplots(nrows=1,ncols=len(panels),proj='cyl',refwidth=3,share=False)
axsf    = np.atleast_1d(axs).ravel()
for ax,(title,arr) in zip(axsf,panels):
    m = ax.pcolormesh(lon,lat,arr,cmap='terrain',vmin=0,vmax=900,levels=30,extend='max',
                      transform=ccrs.PlateCarree())
    ax.format(title=title,latlim=(float(lat.min()),float(lat.max())),
              lonlim=(float(lon.min()),float(lon.max())),
              coast=True,borders=False,grid=True,gridminor=False)
fig.colorbar(m,loc='b',ax=axsf[-1],width=0.15)
pplt.show()
fig.save('../figs/fig_S6.jpg')

In [ ]:
namesorog = [n for n in ['sr_hi','nn_gauss','sr_med','nn_bl'] if n in MODELPRED]
obs        = obsflat[valid]

for orogflat,oroglabel,figname in [
        (sdoraw,'Std dev of orography (m)','fig_S7.jpg'),
        *( [(SEFLAT,'Surface elevation (m)','fig_S8.jpg')] if SEFLAT is not None else [] )]:
    orog  = orogflat[valid]
    lo,hi = np.percentile(orog,[2,98])
    edges = np.linspace(lo,hi,11); xc = 0.5*(edges[:-1]+edges[1:])
    idxs  = np.clip(np.digitize(orog,edges)-1,0,9)
    fig,ax = pplt.subplots(refwidth=3,refheight=2)
    for name in namesorog:
        pred = MODELPRED[name][valid]
        r2s  = [r2(obs[idxs==i],pred[idxs==i]) if (idxs==i).sum()>=500 else np.nan
                for i in range(10)]
        ax.plot(xc,r2s,color=COLORS[name],linewidth=1.5,marker='o',markersize=3,
                label=LABELS[name])
    ax.format(xlabel=oroglabel,ylabel='R$^2$',title=f'Skill vs. {oroglabel}',grid=False)
    ax.legend(loc='ur',ncols=1)
    pplt.show()
    fig.save(f'../figs/{figname}')